# Simple Data Operations — Unity Catalog & BigQuery

In [ ]:
BQ_PROJECT = "idd-dscoe-nonprod-1"
BQ_DATASET = "p17851h"
UC_SCHEMA = "main.p17851h"

## Unity Catalog — Read

In [ ]:
df = spark.table(f"{UC_SCHEMA}.sample_data")
display(df)

## Unity Catalog — Write

In [ ]:
df.write.mode("overwrite").saveAsTable(f"{UC_SCHEMA}.sample_data_copy")

## Unity Catalog — Append

In [ ]:
df.write.mode("append").saveAsTable(f"{UC_SCHEMA}.sample_data_copy")
display(spark.table(f"{UC_SCHEMA}.sample_data_copy"))

## BigQuery — Read

In [ ]:
df_bq = (
    spark.read.format("bigquery")
    .option("table", f"{BQ_PROJECT}.{BQ_DATASET}.sample_data")
    .load()
)
display(df_bq)

## BigQuery — Write

In [ ]:
(
    df_bq.write.format("bigquery")
    .option("writeMethod", "direct")
    .option("table", f"{BQ_DATASET}.spark_write_test")
    .mode("overwrite")
    .save()
)

## BigQuery — Append

In [ ]:
(
    df_bq.write.format("bigquery")
    .option("writeMethod", "direct")
    .option("table", f"{BQ_DATASET}.spark_write_test")
    .mode("append")
    .save()
)

display(
    spark.read.format("bigquery")
    .option("table", f"{BQ_PROJECT}.{BQ_DATASET}.spark_write_test")
    .load()
)

## BigQuery → Unity Catalog

Read from BigQuery, write to Unity Catalog.

In [ ]:
df_bq.write.mode("overwrite").saveAsTable(f"{UC_SCHEMA}.from_bigquery")
display(spark.table(f"{UC_SCHEMA}.from_bigquery"))

## Unity Catalog → BigQuery

Read from Unity Catalog, write to BigQuery.

In [ ]:
df_uc = spark.table(f"{UC_SCHEMA}.sample_data")

(
    df_uc.write.format("bigquery")
    .option("writeMethod", "direct")
    .option("table", f"{BQ_DATASET}.from_unity_catalog")
    .mode("overwrite")
    .save()
)

display(
    spark.read.format("bigquery")
    .option("table", f"{BQ_PROJECT}.{BQ_DATASET}.from_unity_catalog")
    .load()
)

## Cleanup

In [ ]:
spark.sql(f"DROP TABLE IF EXISTS {UC_SCHEMA}.sample_data_copy")
spark.sql(f"DROP TABLE IF EXISTS {UC_SCHEMA}.from_bigquery")